# Memory screen analysis

This notebook analyzes the optional initial memory screen separately from the four experimental blocks. It audits selection, omissions, item and pair performance, response times, and the actual recorded movement. It uses participant exports only.

| Stage | Input | Output |
|---|---|---|
| Discover sessions | Raw CSVs, or clean tables when raw files are unavailable | One session per full source filename |
| Recompute the criterion | 20 completed encoding turns and 10 unique order questions | Correct count and eligibility under the 6 of 10 rule |
| Compare recorded results | Saved summary and independently reconstructed answers | Explicit discrepancy flags |
| Describe performance | Complete screens, including final omissions | Version specific accuracy, response time, and timeout summaries |
| Audit stimulus delivery | Actual positions and saved motion metadata | Geometry, fixed step, direction balance, and timing checks |

The design specification is [06_memory_screen.ipynb](../../design_checks/notebooks/06_memory_screen.ipynb). This notebook evaluates observed delivery and responses. Configuration checks alone cannot establish equal surprise, equal memorability, or a stable memory ability trait.

## Inputs and privacy

Raw files are preferred because the standard splitter's clean filenames contain participant and date but omit source time. Two raw files from the same participant on the same day are therefore processed in separate temporary directories and remain separate sessions. If only clean tables exist, their full file stems define sessions. Any sessions already overwritten by an earlier split cannot be recovered from clean files.

The automatic mode discovers raw CSVs under both `analysis/data/memory_screen/raw/` and the shared full-session directory `analysis/data/drone_task/raw/`, ignoring exact file copies. When no raw files exist, it uses `analysis/data/memory_screen/clean/`. It never combines raw and clean copies. Use an explicit mode if one source collection is incomplete. Repeated sessions from the same person remain repeated sessions, so descriptive denominators below count sessions rather than independent participants.

Source filenames and participant identifiers stay internal. Public notebook displays show aggregate tables. Optional audit exports use ordinal session labels and go under the ignored data directory.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'static/task/memory_screening/memory_screening.js').exists())
sys.path.insert(0, str(ROOT / 'analysis/memory_screen'))
from memory_screen_analysis import (
    load_screen_data, runtime_design, analyze_screen_data, plot_report, export_report
)

INPUT_MODE = 'auto'  # auto, raw, or clean
RAW_DIRS = [ROOT / 'analysis/data/memory_screen/raw', ROOT / 'analysis/data/drone_task/raw']
CLEAN_DIR = ROOT / 'analysis/data/memory_screen/clean'
OUTPUT_DIR = ROOT / 'analysis/data/memory_screen/derived'
WRITE_EXPORTS = True

expected = runtime_design()
display(pd.DataFrame([
    ['Module version', expected['version']],
    ['Series version', expected['series_version']],
    ['Item version', expected['item_version']],
    ['Encoding turns', expected['n_encoding']],
    ['Order questions', expected['n_pairs']],
    ['Correct answer criterion', expected['min_correct']],
    ['Initial drone position', expected['drone_initial_position']],
    ['Absolute shift in rail percentage points', expected['drone_step_size']],
    ['Movement calibration metric', expected['drone_step_metric']],
], columns=['Configuration', 'Value']))

(node:41259) Warning: The 'NO_COLOR' env is ignored due to the 'FORCE_COLOR' env being set.
(Use `node --trace-warnings ...` to show where the warning was created)


,Configuration,Value
0,Module version,ms-3.4.0-2026-10-25
1,Series version,ms-hidden-medium-v5
2,Item version,ms-objects-normed-g2
3,Encoding turns,20
4,Order questions,10
5,Correct answer criterion,6
6,Initial drone position,50
7,Absolute shift in rail percentage points,9.05
8,Movement calibration metric,mean_absolute_nonzero_main_drone_shift


## Session inventory

| Outcome | Interpretation |
|---|---|
| Complete pass | Complete recorded protocol and at least 6 independently verified correct answers |
| Complete fail | Complete recorded protocol and fewer than 6 correct answers |
| Interrupted | Assigned or observed screen without a final summary |
| Configuration error | Screen assets or configuration failed |
| Unverifiable | Required completion or answer key evidence is unavailable |
| Data issue | Duplicate or invalid question records, invalid item coverage, malformed keys, or a source loading error |
| Assignment unknown | No screen records and no usable assignment flag |
| Not requested | Explicitly disabled screen with no screen records |

Complete protocol requires encoding serials 1 through 20 exactly once, all with `completed=1`, question IDs 1 through 10 exactly once, and every encoded item tested exactly once. A missing completion column is not treated as evidence of successful encoding. An interrupted session can have a calculable hypothetical criterion result, but it is not counted as an observed completed pass.

Both pass rates are reported. Passed among assigned retains interrupted and technical outcomes in the denominator. Passed among complete describes selection only after a complete assessment. Sessions with unknown assignment are shown separately.

In [2]:
data = load_screen_data(RAW_DIRS, CLEAN_DIR, mode=INPUT_MODE)
report = analyze_screen_data(data, expected)
display(pd.DataFrame([
    ['Input mode', data['mode']],
    ['Available source sessions', len(data['sources'])],
    ['Exact raw copies ignored', data['duplicate_raw_copies']],
    ['Metadata rows', len(data['metadata'])],
    ['Screen encoding rows', len(data['encoding'])],
    ['Screen question rows', len(data['order'])],
    ['Screen summary rows', len(data['summary'])],
], columns=['Inventory', 'Observed']))

if not report['sessions'].assigned.any():
    print('No memory screen observations are available. No accuracy, response time, or pass rate is estimated.')
display(report['outcomes'])
display(report['rates'].round({'fraction': 3}))

,Inventory,Observed
0,Input mode,raw
1,Available source sessions,2
2,Exact raw copies ignored,0
3,Metadata rows,2
4,Screen encoding rows,0
5,Screen question rows,0
6,Screen summary rows,0


No memory screen observations are available. No accuracy, response time, or pass rate is estimated.


,status,n_sessions,n_assigned,fraction_of_assigned
0,complete_pass,0,0,NaN
1,complete_fail,0,0,NaN
2,interrupted,0,0,NaN
3,configuration_error,0,0,NaN
4,unverifiable,0,0,NaN
5,data_issue,0,0,NaN
6,assignment_unknown,2,0,NaN
7,not_requested,0,0,NaN


,measure,numerator,denominator,fraction
0,Available sessions,2,2,1.0
1,Assigned or observed screen,0,2,0.0
2,Complete screen among assigned,0,0,NaN
3,Passed among assigned,0,0,NaN
4,Passed among complete,0,0,NaN
5,Assigned sessions with audit issues,0,0,NaN


## Saved results and material versions

Correctness is reconstructed from the studied item at the earlier serial position, the actual two displayed images, the selected side, and the final timeout flag. Saved `order_correct_bin` is checked rather than used as the answer key. A final omission counts incorrect. A timeout on the first attempt followed by a valid second response is distinct from a final omission.

A saved count or pass flag that disagrees with a verifiable reconstruction remains an audit discrepancy. It is never silently substituted for reconstructed performance. Invalid displayed pairs, fractional serial positions, reused item endpoints, duplicate questions, and multiple summaries prevent a valid completed protocol. Malformed attempt histories are flagged without crashing the analysis.

Material versions remain separate. Performance should not be pooled across different geometry or item versions without a deliberate decision. A screen result describes this short assessment rather than a diagnostic measure of memory ability.

In [3]:
if report['sessions'].assigned.any():
    display(report['versions'])
    if len(report['audit_issues']):
        display(report['audit_issues'])
    else:
        print('No audit discrepancies were found in the available assigned screens.')
else:
    print('There are no assigned or observed memory screens to audit.')

complete = report['sessions'][report['sessions'].status.isin(['complete_pass', 'complete_fail'])]
if len(complete):
    display(complete.groupby(['module_version', 'series_version', 'item_version']).agg(
        n_complete=('session', 'size'),
        mean_correct=('n_correct', 'mean'),
        median_correct=('n_correct', 'median'),
        mean_final_omissions=('n_final_timeouts', 'mean'),
        mean_questions_with_any_timeout=('n_any_timeouts', 'mean'),
    ).round(3).reset_index())

There are no assigned or observed memory screens to audit.


## Questions, items, and response times

The gap table uses serial distance, so gap 2 means one intervening item and gap 3 means two. The pair table separates the exact studied positions. The item table reports the accuracy of the pair involving each object. A pair error cannot reveal which of its two objects was forgotten.

Accuracy includes final omissions as incorrect. Median answered response time uses the successful response attempt and excludes final omissions. Total question duration includes reminder and retry time. Any timeout and final timeout are shown separately. These are descriptive summaries from complete screens. Questions within one session and two endpoints of the same question are not independent observations.

In [4]:
if len(report['gap_performance']):
    display(report['gap_performance'].round(3))
    display(report['pair_performance'].round(3))
    display(report['item_performance'].round(3))
else:
    print('No complete, verifiable memory screen questions are available for item or pair analysis.')

No complete, verifiable memory screen questions are available for item or pair analysis.


## Motion and delivery audit

Observed drone and supply positions are compared with the arrays saved in that session's summary. When the saved series version matches the current runtime version, they are also checked against the live JavaScript configuration. This avoids applying today's geometry to another material version.

| Controlled quantity | Current implementation | Main reference |
|---|---|---|
| Volatility operational measure | Mean absolute drone shift of 9.05 percentage points per turn | Midpoint of mean absolute nonzero within-block target differences; stationary transitions excluded |
| Stochasticity operational measure | Mean absolute rendered drone-to-landing distance of 2.957285 percentage points | Midpoint of the main blocks' mean absolute rendered distances under both reward/loss assignments |
| Drone direction | Exactly 10 left and 10 right shifts | Includes baseline to turn 1 |
| Supply side | Exactly 10 landings left and 10 right of the drone | An irregular sequence, without an alternation requirement |

The current series is `ms-hidden-medium-v5`, with the drone hidden throughout. The new version distinguishes the larger movement step calibrated from actual main-task position changes. The direction sequences, supply offset magnitude, item pool, and response protocol are retained. The volatility reference excludes 34 stationary transitions from the 49 adjacent differences in each main block and averages the remaining 15 position changes. It also excludes between-block jumps and the unknown first main-block baseline. The main source low-volatility label has a realized mean movement on position changes of 14.099333 percentage points, while the high-volatility label has 4.000600. Their midpoint is 9.049967, rounded to the screening step of 9.05. Source low-stochasticity blocks have mean rendered landing distance 3.38752, while source high-stochasticity blocks have 2.52705. Main labels and arrays are retained; calibration follows measured implementation behavior. The live design notebook derives these comparisons from source arrays and renderer bounds.

Mean absolute distance and residual SD are different measures. Medium stochasticity here means the midpoint by mean absolute rendered distance; it does not assert an intermediate variance. The screen holds absolute offset magnitude constant while varying its side. Equal distances and balanced directions do not establish constant subjective surprise or independent responses.

The constant-step check includes the initial 50 percent baseline and all 20 target positions. Counting only successive targets would yield 19 shifts. The audit verifies equal absolute shifts, 10 left and 10 right, logged starts and signed shifts, logged direction and the retained animation-duration parameter, and the summary baseline, step size, calibration metric, and direction list. The current metric is `mean_absolute_nonzero_main_drone_shift`. The animation parameter is inactive with the drone hidden. Per-turn and summary visibility metadata are also checked against the current hidden-drone configuration.

For complete current-version records, supply checks independently verify the 10/10 side balance and every recorded absolute offset against the live configured size. Per-turn side/size fields, saved wind offsets, and summary size/directions/metric must agree with the actual drone and supply positions. The current configured positions are inside the renderer's bounds, so recorded and rendered distances agree. The level table below reports distances computed from recorded positions.

Older series retain their saved-array audits without imposing this geometry retrospectively. A mismatch is a delivery or provenance concern and is retained as an audit issue; it does not count as evidence of poor memory.

In [5]:
geometry = report['geometry']
if len(geometry) and geometry.geometry_rows.gt(0).any():
    geometry_summary = geometry.groupby('series_version', dropna=False).agg(
        n_sessions=('session', 'nunique'),
        n_saved_geometry_checked=('saved_geometry_matches', 'count'),
        n_saved_geometry_mismatches=('saved_geometry_matches', lambda x: x.eq(False).sum()),
        n_current_geometry_checked=('current_geometry_matches', 'count'),
        n_current_geometry_mismatches=('current_geometry_matches', lambda x: x.eq(False).sum()),
        n_movement_log_checked=('movement_log_matches', 'count'),
        n_movement_log_mismatches=('movement_log_matches', lambda x: x.eq(False).sum()),
        n_summary_motion_checked=('summary_motion_matches', 'count'),
        n_summary_motion_mismatches=('summary_motion_matches', lambda x: x.eq(False).sum()),
        n_constant_step_failures=('constant_step_contract', lambda x: x.eq(False).sum()),
    ).reset_index()
    display(geometry_summary)
    supply_levels = geometry.groupby('series_version', dropna=False).agg(
        n_sessions=('session', 'nunique'),
        median_mean_abs_distance=('mean_abs_residual', 'median'),
        median_residual_sd=('residual_sd', 'median'),
        median_signed_distance=('residual_mean', 'median'),
        median_left_landings=('supply_n_left', 'median'),
        median_right_landings=('supply_n_right', 'median'),
    ).reset_index()
    display(supply_levels.round(6))
    checks = {
        'Supply side balance': 'supply_balance_contract',
        'Absolute offset magnitude': 'offset_magnitude_contract',
        'Per-turn supply metadata': 'supply_log_matches',
        'Summary supply metadata': 'summary_supply_matches',
    }
    noise_audit = []
    for version, rows in geometry.groupby('series_version', dropna=False):
        for check, field in checks.items():
            noise_audit.append({'series_version': version, 'check': check,
                'n_checked': int(rows[field].notna().sum()),
                'n_mismatches': int(rows[field].eq(False).sum())})
    display(pd.DataFrame(noise_audit))
else:
    print('No recorded screening trajectories are available to compare.')


No recorded screening trajectories are available to compare.


## Figures and exports

Figures appear only when assigned screen observations exist. The outcome chart retains technical and incomplete sessions. The score histogram uses complete screens. Question plots remain separated by material version.

CSV audits and optional PNG figures are written under `analysis/data/memory_screen/derived/`. Source participant IDs and source filenames are not written to these reports. The ordinal labels allow a researcher to connect audit tables within this run without publishing the original identifiers.

In [6]:
figures = plot_report(report)
if figures:
    import matplotlib.pyplot as plt
    for name, figure in figures:
        display(figure)
        if WRITE_EXPORTS:
            OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
            figure.savefig(OUTPUT_DIR / f'{name}.png', dpi=180, bbox_inches='tight')
        plt.close(figure)
else:
    print('No behavioral figures were generated because no assigned screen observations are available.')

if WRITE_EXPORTS:
    written = export_report(report, OUTPUT_DIR)
    display(pd.DataFrame({'Exported audit table': written}))

No behavioral figures were generated because no assigned screen observations are available.


,Exported audit table
0,sessions.csv
1,outcomes.csv
2,rates.csv
3,audit_issues.csv
4,versions.csv
5,questions.csv
6,pair_performance.csv
7,gap_performance.csv
8,item_performance.csv
9,geometry.csv


## Interpretation and checks

The retained criterion is at least 6 correct out of 10. Under independent random binary guesses, approximately 37.7 percent pass. The screen therefore provides a permissive selection rule, not a reliable separation of good and poor memory traits. Actual responses may also share attention, task familiarity, or episode-level influences.

Report assignment, technical failures, interruptions, and complete pass and fail counts together. Keep the screen outcome separate from the study's experimental memory outcomes. Do not optimize the threshold or item set using the same main task outcomes that the final study will test.

To run this notebook, install `design_checks/requirements-memory-screen.txt`, select that Python environment, and execute all cells. The notebook reads the current source design through Node.